<a href="https://colab.research.google.com/github/Coobaltoo13/milling-tool-life-fusion/blob/main/notebooks/%2006_Validaci%C3%B3n_rigurosa_y_baselines.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Validación Rigurosa: Train/Validation/Test + Múltiples Semillas + Baselines

**Objetivo:** Corregir fuga de información, añadir baselines y validar con múltiples semillas.

In [1]:
#Librerias
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.metrics import f1_score, precision_score, recall_score, roc_auc_score
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks

print("TensorFlow:", tf.__version__)

TensorFlow: 2.20.0


In [19]:
url = 'https://raw.githubusercontent.com/Coobaltoo13/milling-tool-life-fusion/main/data/raw/FeatureAndMetadata_Milling.csv'
df = pd.read_csv(url, sep=';', header=0, decimal=',', skiprows=[1])

# X = primeras 120 columnas (la 121 estaba vacía)
X = df.iloc[:, :120].apply(pd.to_numeric, errors='coerce')

# y = Column131 (CycleToFailureNormalized, valores 0.0 a 1.0)
y = pd.to_numeric(df['Column131'], errors='coerce')

# DIAGNÓSTICO
print("Min y:", y.min(), "| Max y:", y.max())
print("Primeros valores de y:", y.head(5).tolist())

# Limpiar
X = X.fillna(X.mean())
mask = ~y.isnull()
X = X[mask].values
y = y[mask].values

# Dividir 60/20/20
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.4, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

# Normalizar (fit SOLO en train)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

print("Train:", X_train_scaled.shape, "| y_train min/max:", y_train.min(), y_train.max())
print("Val:", X_val_scaled.shape)
print("Test:", X_test_scaled.shape)

Min y: 0.0 | Max y: 1.0
Primeros valores de y: [1.0, 0.979591837, 0.959183673, 0.93877551, 0.918367347]
Train: (580, 120) | y_train min/max: 0.0 1.0
Val: (194, 120)
Test: (194, 120)


In [21]:
#Funcion de metricas
def calcular_metricas(y_true, y_pred):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    y_true_class = (y_true >= 0.5).astype(int)
    y_pred_class = (y_pred >= 0.5).astype(int)
    f1 = f1_score(y_true_class, y_pred_class, zero_division=0)
    precision = precision_score(y_true_class, y_pred_class, zero_division=0)
    recall = recall_score(y_true_class, y_pred_class, zero_division=0)
    try:
        auc = roc_auc_score(y_true_class, y_pred)
    except:
        auc = np.nan
    return rmse, mae, r2, f1, precision, recall, auc

In [22]:
#Funciones de modelos
def build_autoencoder(input_dim, latent_dim):
    encoder_input = layers.Input(shape=(input_dim,))
    x = layers.Dense(64, activation='relu')(encoder_input)
    x = layers.Dense(32, activation='relu')(x)
    latent = layers.Dense(latent_dim, activation='relu', name='latent')(x)
    x = layers.Dense(32, activation='relu')(latent)
    x = layers.Dense(64, activation='relu')(x)
    decoder_output = layers.Dense(input_dim, activation='linear')(x)
    autoencoder = models.Model(encoder_input, decoder_output)
    autoencoder.compile(optimizer='adam', loss='mse')
    encoder = models.Model(encoder_input, latent)
    return autoencoder, encoder

def build_lstm(latent_dim, seed=None):
    if seed is not None:
        tf.keras.utils.set_random_seed(seed)
    model = models.Sequential([
        layers.Input(shape=(1, latent_dim)),
        layers.LSTM(32, return_sequences=False),
        layers.Dense(16, activation='relu'),
        layers.Dense(1, activation='linear')
    ])
    model.compile(optimizer='adam', loss='mse', metrics=['mae'])
    return model

In [23]:
#Analisis de sensibilidad usando VALIDACION
latent_dims = [2, 4, 8, 16, 24, 32, 48, 64, 80]
resultados_val = []

for dim in latent_dims:
    print(f"Dim: {dim}")
    ae, encoder = build_autoencoder(X_train_scaled.shape[1], dim)
    ae.fit(X_train_scaled, X_train_scaled, epochs=30, batch_size=32,
           validation_split=0.1, verbose=0)

    X_train_latent = encoder.predict(X_train_scaled, verbose=0)
    X_val_latent = encoder.predict(X_val_scaled, verbose=0)

    X_train_lstm = X_train_latent.reshape((X_train_latent.shape[0], 1, dim))
    X_val_lstm = X_val_latent.reshape((X_val_latent.shape[0], 1, dim))

    lstm = build_lstm(dim)
    lstm.fit(X_train_lstm, y_train, epochs=30, batch_size=32,
             validation_split=0.1, verbose=0)

    y_pred_val = lstm.predict(X_val_lstm, verbose=0).flatten()
    rmse, mae, r2, f1, precision, recall, auc = calcular_metricas(y_val, y_pred_val)

    resultados_val.append({'dim': dim, 'R2': r2, 'RMSE': rmse, 'MAE': mae,
                           'F1': f1, 'AUC': auc})

df_val = pd.DataFrame(resultados_val)
print(df_val)

# Mejor dimensión según VALIDACIÓN
best_dim = int(df_val.loc[df_val['R2'].idxmax(), 'dim'])
print(f"\nMejor dimensión según validación: {best_dim}")

Dim: 2
Dim: 4
Dim: 8
Dim: 16
Dim: 24
Dim: 32
Dim: 48
Dim: 64
Dim: 80
   dim        R2      RMSE       MAE        F1       AUC
0    2  0.294560  0.259800  0.213921  0.744186  0.759247
1    4  0.471624  0.224844  0.179571  0.796296  0.849277
2    8  0.630885  0.187927  0.152804  0.841584  0.916773
3   16  0.640555  0.185449  0.144641  0.857143  0.929209
4   24  0.605268  0.194339  0.147738  0.837696  0.901148
5   32  0.579275  0.200636  0.155559  0.818653  0.903912
6   48  0.680656  0.174799  0.135117  0.832487  0.932079
7   64  0.638116  0.186077  0.142930  0.843137  0.897215
8   80  0.682745  0.174226  0.132916  0.855670  0.921237

Mejor dimensión según validación: 80


In [24]:
#Evaluar mejor moelo en TEST (solo una vez)
# Entrenar autoencoder final con la mejor dimensión
ae_final, encoder_final = build_autoencoder(X_train_scaled.shape[1], best_dim)
ae_final.fit(X_train_scaled, X_train_scaled, epochs=50, batch_size=32,
             validation_split=0.1, verbose=0)

X_train_latent = encoder_final.predict(X_train_scaled, verbose=0)
X_test_latent = encoder_final.predict(X_test_scaled, verbose=0)

X_train_lstm = X_train_latent.reshape((X_train_latent.shape[0], 1, best_dim))
X_test_lstm = X_test_latent.reshape((X_test_latent.shape[0], 1, best_dim))

lstm_final = build_lstm(best_dim)
lstm_final.fit(X_train_lstm, y_train, epochs=50, batch_size=32,
               validation_split=0.1, verbose=0)

y_pred_test = lstm_final.predict(X_test_lstm, verbose=0).flatten()
rmse, mae, r2, f1, precision, recall, auc = calcular_metricas(y_test, y_pred_test)

print("=" * 50)
print(f"RESULTADO FINAL EN TEST (dim={best_dim})")
print("=" * 50)
print(f"RMSE: {rmse:.4f}")
print(f"MAE:  {mae:.4f}")
print(f"R²:   {r2:.4f}")
print(f"F1:   {f1:.4f}")
print(f"AUC:  {auc:.4f}")

RESULTADO FINAL EN TEST (dim=80)
RMSE: 0.1729
MAE:  0.1210
R²:   0.6294
F1:   0.8677
AUC:  0.9381


In [25]:
#Multiples semillas
seeds = [1, 2, 3, 4, 5]
resultados_seeds = []

for seed in seeds:
    print(f"Seed: {seed}")
    tf.keras.utils.set_random_seed(seed)

    ae, encoder = build_autoencoder(X_train_scaled.shape[1], best_dim)
    ae.fit(X_train_scaled, X_train_scaled, epochs=50, batch_size=32,
           validation_split=0.1, verbose=0)

    X_train_latent = encoder.predict(X_train_scaled, verbose=0)
    X_test_latent = encoder.predict(X_test_scaled, verbose=0)

    X_train_lstm = X_train_latent.reshape((X_train_latent.shape[0], 1, best_dim))
    X_test_lstm = X_test_latent.reshape((X_test_latent.shape[0], 1, best_dim))

    lstm = build_lstm(best_dim, seed=seed)
    lstm.fit(X_train_lstm, y_train, epochs=50, batch_size=32,
             validation_split=0.1, verbose=0)

    y_pred = lstm.predict(X_test_lstm, verbose=0).flatten()
    rmse, mae, r2, f1, precision, recall, auc = calcular_metricas(y_test, y_pred)
    resultados_seeds.append({'seed': seed, 'R2': r2, 'RMSE': rmse, 'MAE': mae,
                             'F1': f1, 'AUC': auc})

df_seeds = pd.DataFrame(resultados_seeds)
print(df_seeds)
print("\nPromedio ± Desviación estándar:")
print(df_seeds[['R2','RMSE','MAE','F1','AUC']].agg(['mean','std']).round(4))

Seed: 1
Seed: 2
Seed: 3
Seed: 4
Seed: 5
   seed        R2      RMSE       MAE        F1       AUC
0     1  0.703990  0.154515  0.107983  0.861538  0.934078
1     2  0.675361  0.161814  0.110422  0.891304  0.953854
2     3  0.681582  0.160256  0.110995  0.882051  0.951940
3     4  0.669486  0.163272  0.117395  0.875000  0.933759
4     5  0.672140  0.162615  0.119856  0.876289  0.941627

Promedio ± Desviación estándar:
          R2    RMSE     MAE      F1     AUC
mean  0.6805  0.1605  0.1133  0.8772  0.9431
std   0.0139  0.0035  0.0050  0.0109  0.0095


In [26]:
#Baselines Simples
baselines = {
    'Linear Regression': LinearRegression(),
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42),
    'XGBoost': XGBRegressor(n_estimators=100, random_state=42, verbosity=0)
}

resultados_baselines = []

for nombre, modelo in baselines.items():
    modelo.fit(X_train_scaled, y_train)
    y_pred = modelo.predict(X_test_scaled)
    rmse, mae, r2, f1, precision, recall, auc = calcular_metricas(y_test, y_pred)
    resultados_baselines.append({'modelo': nombre, 'R2': r2, 'RMSE': rmse,
                                 'MAE': mae, 'F1': f1, 'AUC': auc})

df_baselines = pd.DataFrame(resultados_baselines)
print(df_baselines)

              modelo        R2      RMSE       MAE        F1       AUC
0  Linear Regression  0.397473  0.220447  0.127911  0.847291  0.921850
1      Random Forest  0.837290  0.114557  0.084343  0.900524  0.972142
2            XGBoost  0.847088  0.111055  0.074519  0.905263  0.976077


In [29]:
# Promedio del LSTM de las 5 semillas (de la Celda 8)
lstm_mean = df_seeds[['R2', 'RMSE', 'MAE', 'F1', 'AUC']].mean().round(4).to_dict()

resultado_lstm = {
    'modelo': f'Autoencoder + LSTM (dim={best_dim}, mean 5 seeds)',
    'R2': lstm_mean['R2'],
    'RMSE': lstm_mean['RMSE'],
    'MAE': lstm_mean['MAE'],
    'F1': lstm_mean['F1'],
    'AUC': lstm_mean['AUC']
}

comparacion_final = pd.concat([
    df_baselines,
    pd.DataFrame([resultado_lstm])
], ignore_index=True)

print(comparacion_final)

                                      modelo        R2      RMSE       MAE  \
0                          Linear Regression  0.397473  0.220447  0.127911   
1                              Random Forest  0.837290  0.114557  0.084343   
2                                    XGBoost  0.847088  0.111055  0.074519   
3  Autoencoder + LSTM (dim=80, mean 5 seeds)  0.680500  0.160500  0.113300   

         F1       AUC  
0  0.847291  0.921850  
1  0.900524  0.972142  
2  0.905263  0.976077  
3  0.877200  0.943100  


In [28]:
print("Min y:", y.min())
print("Max y:", y.max())
print("Primeros valores:", y[:10])

Min y: 0.0
Max y: 1.0
Primeros valores: [1.         0.97959184 0.95918367 0.93877551 0.91836735 0.89795918
 0.87755102 0.85714286 0.83673469 0.81632653]


In [17]:
# Ver las últimas 4 columnas y sus valores min/max
for col in df.columns[-4:]:
    try:
        print(f"{col}: min={df[col].min()}, max={df[col].max()}, primeros={df[col].dropna().head(3).tolist()}")
    except:
        print(f"{col}: no numérica")

Column128: min=35.33, max=41.67, primeros=[36.67, 36.67, 36.67]
Column129: min=80, max=160, primeros=[80, 80, 80]
Column130: min=0, max=149, primeros=[49, 48, 47]
Column131: min=0.0, max=1.0, primeros=[1.0, 0.979591837, 0.959183673]


In [18]:
print("Min y:", y.min(), "| Max y:", y.max())

Min y: 1 | Max y: 150


## Conclusión

Se implementó una validación rigurosa con train (60%) / validation (20%) / test (20%). La dimensión óptima se seleccionó usando validation, y el test se usó una sola vez para reportar el resultado final. Se repitió con 5 semillas y se comparó contra baselines.

**Resultados en test:**

| Modelo | R² | RMSE | MAE | F1 | AUC |
|:---|:---|:---|:---|:---|:---|
| Linear Regression | 0.397 | 0.220 | 0.128 | 0.847 | 0.922 |
| Random Forest | 0.837 | 0.115 | 0.084 | 0.901 | 0.972 |
| **XGBoost** | **0.847** | **0.111** | **0.075** | **0.905** | **0.976** |
| Autoencoder + LSTM (dim=80) | 0.681 ± 0.014 | 0.161 ± 0.004 | 0.113 ± 0.005 | 0.877 ± 0.011 | 0.943 ± 0.010 |

**Hallazgos:**

1. **XGBoost supera al autoencoder + LSTM** en este dataset (R² = 0.847 vs 0.681).
2. **Random Forest es el segundo mejor**, muy cercano a XGBoost.
3. **El LSTM solo supera a regresión lineal**, no a los métodos de ensamble.
4. **La validación rigurosa cambió el resultado:** en los notebooks 04-06, el LSTM parecía mejor (R²=0.78) porque la dimensión se eligió usando test. Al hacerlo correctamente, baja a 0.68.

**Interpretación:** Este resultado es consistente con la literatura sobre datos tabulares: con ~1000 filas y 120 features, los árboles (Random Forest, XGBoost) suelen superar a las redes neuronales. El autoencoder + LSTM es más complejo y necesita más datos para aprovechar su capacidad.

**Implicación práctica:** Para aplicaciones de mantenimiento predictivo en producción sostenible con datasets de tamaño similar, se recomienda empezar con XGBoost como baseline antes de invertir en deep learning.